# Notebook 04c — SHAP Recompute on Canonical Shared Samples

**Purpose:** Re-run SHAP for all 18 5-class models using deterministic per-dataset canonical sample indices, so all 3 architectures within each dataset evaluate on the SAME test samples. Required for cross-model agreement analysis (Notebook 06) and any downstream sample-level comparison.

**Root cause being fixed:**
Notebook 04 derived its random seed per-model from `SEED + abs(hash(model_name)) % 1000`, which meant different architectures sampled different 1000 stratified test indices. Cross-model overlap was small (~100-235 samples per pair, dominated by rare classes). This made Krishna et al. cross-architecture agreement metrics meaningless.

**Fix:** Hoist sample selection out of the per-model loop. Each dataset gets ONE canonical 1000-index test set and ONE canonical 50-index background set. All 3 architectures use these.

**Methodology preservation:** Identical SHAP computation logic to Notebook 04. Same explainers (TreeExplainer for RF/XGB, GradientExplainer for DNN), same stratified sampling function, same shape standardization, same model loading. Only the random seed derivation changes.

**Output files (do NOT overwrite Notebook 04 outputs):**
- `shap_values/{dataset}/{model_name}_shap_shared.npy` — SHAP arrays on canonical samples
- `shap_values/{dataset}/{model_name}_eval_idx_shared.npy` — canonical eval indices (same per dataset)
- `shap_values/{dataset}/{model_name}_shap_shared_meta.json` — metadata per model

**Resumability:** Skip if `_shap_shared.npy` already exists. Designed to handle Drive disconnect recovery without recomputing completed models.

**Time estimate:** ~90 min total wall clock on T4 GPU. Same scale as original Notebook 04.

## 1. Setup

In [1]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

import os, shutil
REPO = '/content/drive/MyDrive/XIDS_Research/xids-research'
os.chdir(REPO)

for f in ['.gitconfig', '.git-credentials']:
    src = f'/content/drive/MyDrive/XIDS_Research/{f}'
    if os.path.exists(src):
        shutil.copy(src, f'/root/{f}')
        if f == '.git-credentials':
            os.chmod(f'/root/{f}', 0o600)

print(f'✓ Ready in: {os.getcwd()}')

Mounted at /content/drive
✓ Ready in: /content/drive/MyDrive/XIDS_Research/xids-research


In [2]:
import numpy as np
import json
import joblib
import time
import traceback
from pathlib import Path
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

import torch
import torch.nn as nn
import shap

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'PyTorch device: {DEVICE}')

SEED = 42  # Canonical seed for all dataset sample selections
N_BACKGROUND = 50
N_EVAL = 1000

DATASETS = ['nsl_kdd_v2', 'unsw_nb15_v2', 'cic_ids2017_v2']
MODELS_PER_DATASET = ['rf_5class_smote', 'xgb_5class_smote', 'dnn_5class_smote',
                     'rf_5class_cw', 'xgb_5class_cw', 'dnn_5class_cw']
VARIANTS = ['5class_cw', '5class_smote']

print(f'\nScope: {len(DATASETS)} datasets × {len(MODELS_PER_DATASET)} models = {len(DATASETS) * len(MODELS_PER_DATASET)} SHAP runs')
print(f'N_EVAL: {N_EVAL} (canonical per dataset)')
print(f'N_BACKGROUND: {N_BACKGROUND} (canonical per dataset)')

PyTorch device: cuda

Scope: 3 datasets × 6 models = 18 SHAP runs
N_EVAL: 1000 (canonical per dataset)
N_BACKGROUND: 50 (canonical per dataset)


## 2. Helpers (verbatim from Notebook 04 where possible)

In [3]:
def stratified_sample(X, y, n, rng_local):
    """Identical to Notebook 04 stratified_sample."""
    n_per_class = max(1, n // len(np.unique(y)))
    idx_list = []
    for c in np.unique(y):
        class_idx = np.where(y == c)[0]
        take = min(n_per_class, len(class_idx))
        idx_list.append(rng_local.choice(class_idx, size=take, replace=False))
    idx = np.concatenate(idx_list)
    if len(idx) < n:
        remaining = list(set(range(len(y))) - set(idx.tolist()))
        extra = rng_local.choice(remaining, size=min(n - len(idx), len(remaining)), replace=False)
        idx = np.concatenate([idx, extra])
    return np.sort(idx[:n])

def find_tree_model_path(dataset, model_name):
    base = Path(REPO) / 'models' / dataset
    for ext in ['.pkl', '.joblib']:
        p = base / f'{model_name}{ext}'
        if p.exists():
            return p
    raise FileNotFoundError(f'No tree model file for {model_name} in {base}')

def find_dnn_path(dataset, model_name):
    p = Path(REPO) / 'models' / dataset / f'{model_name}.pt'
    if p.exists():
        return p
    raise FileNotFoundError(f'No DNN file at {p}')

class DNN(nn.Module):
    """Identical architecture to Notebook 04."""
    def __init__(self, in_dim, n_classes, hidden=(256, 128, 64, 32), dropout=0.3):
        super().__init__()
        layers = []
        prev = in_dim
        for h in hidden:
            layers.append(nn.Linear(prev, h))
            layers.append(nn.BatchNorm1d(h))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(dropout))
            prev = h
        layers.append(nn.Linear(prev, n_classes))
        self.net = nn.Sequential(*layers)
    def forward(self, x):
        return self.net(x)

class DNNWithSoftmax(nn.Module):
    """Wraps DNN to output softmax probabilities for SHAP."""
    def __init__(self, dnn):
        super().__init__()
        self.dnn = dnn
    def forward(self, x):
        return torch.softmax(self.dnn(x), dim=-1)

def load_pytorch_dnn(path):
    """Identical to Notebook 04: handles both wrapped checkpoint and bare state dict."""
    ckpt = torch.load(path, map_location=DEVICE, weights_only=False)
    if isinstance(ckpt, dict) and 'state_dict' in ckpt:
        in_dim = ckpt['in_dim']
        n_classes = ckpt['n_classes']
        hidden = tuple(ckpt['hidden'])
        dropout = ckpt['dropout']
        state_dict = ckpt['state_dict']
    else:
        state_dict = ckpt
        first_linear = state_dict['net.0.weight']
        in_dim = first_linear.shape[1]
        last_linear = state_dict['net.16.weight']
        n_classes = last_linear.shape[0]
        hidden = (256, 128, 64, 32)
        dropout = 0.3
    model = DNN(in_dim=in_dim, n_classes=n_classes, hidden=hidden, dropout=dropout)
    model.load_state_dict(state_dict)
    model = model.to(DEVICE)
    model.eval()
    wrapped = DNNWithSoftmax(model).to(DEVICE)
    wrapped.eval()
    return wrapped

print('✓ Helpers loaded')

✓ Helpers loaded


## 3. Compute canonical indices per dataset (the key change vs Notebook 04)

In [4]:
# Canonical indices per dataset: same SEED, no per-model jitter
canonical_indices = {}  # {dataset: {'eval_idx': arr, 'bg_idx': arr}}

for ds in DATASETS:
    PROCESSED = Path(REPO) / 'data' / 'processed' / ds
    y_test = np.load(PROCESSED / 'y_test_5class.npy')
    y_calib = np.load(PROCESSED / 'y_calib_5class.npy')

    # Use SAME seed for every architecture, ONE selection per dataset
    rng = np.random.default_rng(SEED)
    bg_idx = stratified_sample(np.zeros((len(y_calib), 1)), y_calib, N_BACKGROUND, rng)
    eval_idx = stratified_sample(np.zeros((len(y_test), 1)), y_test, N_EVAL, rng)

    canonical_indices[ds] = {'eval_idx': eval_idx, 'bg_idx': bg_idx}

    # Show per-class composition
    from collections import Counter
    eval_class_counts = Counter(y_test[eval_idx].tolist())
    bg_class_counts = Counter(y_calib[bg_idx].tolist())
    print(f'\n{ds}:')
    print(f'  eval_idx: n={len(eval_idx)}, per-class={dict(sorted(eval_class_counts.items()))}')
    print(f'  bg_idx:   n={len(bg_idx)},  per-class={dict(sorted(bg_class_counts.items()))}')
    print(f'  eval first 5: {eval_idx[:5].tolist()}, last 5: {eval_idx[-5:].tolist()}')

# Save canonical indices once (these are dataset-level, not per-model)
for ds, indices in canonical_indices.items():
    out_dir = Path(REPO) / 'shap_values' / ds
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / 'canonical_eval_idx.npy', indices['eval_idx'])
    np.save(out_dir / 'canonical_bg_idx.npy', indices['bg_idx'])
print('\n✓ Canonical indices saved to shap_values/{ds}/canonical_*.npy')


nsl_kdd_v2:
  eval_idx: n=1000, per-class={0: 261, 1: 248, 2: 211, 3: 213, 4: 67}
  bg_idx:   n=50,  per-class={0: 10, 1: 10, 2: 10, 3: 10, 4: 10}
  eval first 5: [36, 37, 57, 60, 69], last 5: [22473, 22496, 22521, 22524, 22532]

unsw_nb15_v2:
  eval_idx: n=1000, per-class={0: 200, 1: 200, 2: 200, 3: 200, 4: 200}
  bg_idx:   n=50,  per-class={0: 10, 1: 10, 2: 10, 3: 10, 4: 10}
  eval first 5: [274, 282, 284, 322, 344], last 5: [63049, 63183, 63324, 63426, 63446]

cic_ids2017_v2:
  eval_idx: n=1000, per-class={0: 354, 1: 230, 2: 209, 3: 200, 4: 7}
  bg_idx:   n=50,  per-class={0: 13, 1: 10, 2: 10, 3: 10, 4: 7}
  eval first 5: [60, 127, 156, 188, 275], last 5: [39704, 39779, 39882, 39896, 39981]

✓ Canonical indices saved to shap_values/{ds}/canonical_*.npy


## 4. Per-model SHAP computation (uses canonical indices, otherwise identical to Notebook 04)

In [5]:
def compute_shap_canonical(dataset, model_name, eval_idx, bg_idx):
    """SHAP computation using CANONICAL eval_idx and bg_idx (not per-model random).
    Otherwise identical methodology to Notebook 04 compute_shap_for_model.
    """
    t0 = time.time()

    PROCESSED = Path(REPO) / 'data' / 'processed' / dataset
    X_test = np.load(PROCESSED / 'X_test.npy').astype(np.float32)
    X_calib = np.load(PROCESSED / 'X_calib.npy').astype(np.float32)

    n_classes = 5
    n_features = X_test.shape[1]

    X_bg = X_calib[bg_idx]
    X_eval = X_test[eval_idx]

    model_type = 'rf' if 'rf' in model_name else ('xgb' if 'xgb' in model_name else 'dnn')

    if model_type in ('rf', 'xgb'):
        path = find_tree_model_path(dataset, model_name)
        model = joblib.load(path)
        explainer = shap.TreeExplainer(model)
        shap_values = explainer.shap_values(X_eval, check_additivity=False)

        if isinstance(shap_values, list):
            shap_values = np.stack(shap_values, axis=-1)
        elif shap_values.ndim == 3:
            if shap_values.shape[0] == n_classes:
                shap_values = np.transpose(shap_values, (1, 2, 0))
            elif shap_values.shape[1] == n_classes:
                shap_values = np.transpose(shap_values, (0, 2, 1))

    elif model_type == 'dnn':
        path = find_dnn_path(dataset, model_name)
        wrapped_model = load_pytorch_dnn(path)

        bg_tensor = torch.from_numpy(X_bg).to(DEVICE)
        eval_tensor = torch.from_numpy(X_eval).to(DEVICE)

        explainer = shap.GradientExplainer(wrapped_model, bg_tensor)
        shap_values = explainer.shap_values(eval_tensor)

        if isinstance(shap_values, list):
            shap_values = np.stack(shap_values, axis=-1)
        elif shap_values.ndim == 3:
            if shap_values.shape[0] == n_classes:
                shap_values = np.transpose(shap_values, (1, 2, 0))
            elif shap_values.shape[1] == n_classes:
                shap_values = np.transpose(shap_values, (0, 2, 1))

        del wrapped_model, bg_tensor, eval_tensor
        if DEVICE == 'cuda':
            torch.cuda.empty_cache()

    expected_shape = (N_EVAL, n_features, n_classes)
    if shap_values.shape != expected_shape:
        if shap_values.shape == (n_classes, N_EVAL, n_features):
            shap_values = np.transpose(shap_values, (1, 2, 0))
        elif shap_values.shape == (N_EVAL, n_classes, n_features):
            shap_values = np.transpose(shap_values, (0, 2, 1))

    t_elapsed = time.time() - t0

    meta = {
        'dataset': dataset,
        'model': model_name,
        'model_type': model_type,
        'shap_shape': list(shap_values.shape),
        'n_background': len(bg_idx),
        'n_eval': len(eval_idx),
        'n_features': n_features,
        'n_classes': n_classes,
        'time_seconds': round(t_elapsed, 1),
        'timestamp': datetime.now().isoformat(),
        'explainer': 'TreeExplainer' if model_type in ('rf', 'xgb') else 'GradientExplainer',
        'shap_target': 'probabilities (softmax)' if model_type == 'dnn' else 'model output (tree)',
        'sample_selection': 'canonical_shared_per_dataset',
        'note': 'SHAP computed on canonical per-dataset shared sample indices. All 3 architectures within a dataset use the same eval_idx and bg_idx. Required for cross-model agreement analysis (Notebook 06).',
    }

    # Save with _shared suffix
    out_dir = Path(REPO) / 'shap_values' / dataset
    out_dir.mkdir(parents=True, exist_ok=True)
    np.save(out_dir / f'{model_name}_shap_shared.npy', shap_values)
    np.save(out_dir / f'{model_name}_eval_idx_shared.npy', eval_idx)
    with open(out_dir / f'{model_name}_shap_shared_meta.json', 'w') as f:
        json.dump(meta, f, indent=2)

    return meta, shap_values.shape, t_elapsed

print('✓ compute_shap_canonical defined')

✓ compute_shap_canonical defined


## 5. Main loop with resumability

In [6]:
results = []
errors = []
t_overall = time.time()

print(f'\n{"="*70}')
print(f'SHAP recompute on canonical samples — {datetime.now().strftime("%H:%M:%S")}')
print(f'{"="*70}\n')

for ds in DATASETS:
    print(f'\n=== {ds} ===')
    eval_idx = canonical_indices[ds]['eval_idx']
    bg_idx = canonical_indices[ds]['bg_idx']

    for model_name in MODELS_PER_DATASET:
        # Resumability check
        out_path = Path(REPO) / 'shap_values' / ds / f'{model_name}_shap_shared.npy'
        idx_path = Path(REPO) / 'shap_values' / ds / f'{model_name}_eval_idx_shared.npy'

        if out_path.exists() and idx_path.exists():
            try:
                existing_shape = np.load(out_path).shape
                existing_idx = np.load(idx_path)
                # Verify it actually uses the canonical indices
                if existing_shape[0] == N_EVAL and np.array_equal(existing_idx, eval_idx):
                    print(f'  ⏭  {model_name:<22} (already done with canonical indices, skipping)')
                    results.append({'dataset': ds, 'model': model_name, 'status': 'skipped'})
                    continue
                else:
                    print(f'  ⚠ {model_name:<22} cached version mismatch, recomputing')
            except Exception:
                pass

        print(f'  ▶  {model_name:<22} ', end='', flush=True)
        try:
            meta, shape, elapsed = compute_shap_canonical(ds, model_name, eval_idx, bg_idx)
            results.append(meta)
            print(f'shape={shape}, {elapsed:.1f}s')
        except Exception as e:
            print(f'❌ FAILED: {type(e).__name__}: {e}')
            errors.append({
                'dataset': ds, 'model': model_name,
                'error': str(e), 'tb': traceback.format_exc(),
            })

elapsed_total = (time.time() - t_overall) / 60
print(f'\nTotal time: {elapsed_total:.1f} min')
print(f'Completed: {len(results)}, Failed: {len(errors)}')


SHAP recompute on canonical samples — 17:28:27


=== nsl_kdd_v2 ===
  ▶  rf_5class_smote        shape=(1000, 122, 5), 166.8s
  ▶  xgb_5class_smote       shape=(1000, 122, 5), 13.9s
  ▶  dnn_5class_smote       shape=(1000, 122, 5), 318.6s
  ▶  rf_5class_cw           shape=(1000, 122, 5), 130.6s
  ▶  xgb_5class_cw          shape=(1000, 122, 5), 8.9s
  ▶  dnn_5class_cw          shape=(1000, 122, 5), 314.3s

=== unsw_nb15_v2 ===
  ▶  rf_5class_smote        shape=(1000, 194, 5), 1636.0s
  ▶  xgb_5class_smote       shape=(1000, 194, 5), 2.1s
  ▶  dnn_5class_smote       shape=(1000, 194, 5), 333.1s
  ▶  rf_5class_cw           shape=(1000, 194, 5), 1361.5s
  ▶  xgb_5class_cw          shape=(1000, 194, 5), 1.9s
  ▶  dnn_5class_cw          shape=(1000, 194, 5), 345.6s

=== cic_ids2017_v2 ===
  ▶  rf_5class_smote        shape=(1000, 78, 5), 121.6s
  ▶  xgb_5class_smote       shape=(1000, 78, 5), 0.9s
  ▶  dnn_5class_smote       shape=(1000, 78, 5), 322.7s
  ▶  rf_5class_cw           shape=(1000,

## 6. Verify canonical alignment across all 18 models

In [ ]:
print('=' * 70)
print('CANONICAL ALIGNMENT VERIFICATION')
print('=' * 70)

all_aligned = True
for ds in DATASETS:
    canonical_eval = canonical_indices[ds]['eval_idx']
    print(f'\n{ds}:')

    for model_name in MODELS_PER_DATASET:
        idx_path = Path(REPO) / 'shap_values' / ds / f'{model_name}_eval_idx_shared.npy'
        if not idx_path.exists():
            print(f'  ❌ {model_name:<22} eval_idx_shared.npy missing')
            all_aligned = False
            continue

        saved_idx = np.load(idx_path)
        matches = np.array_equal(saved_idx, canonical_eval)
        status = '✓' if matches else '❌'
        print(f'  {status} {model_name:<22} matches canonical: {matches}')
        if not matches:
            all_aligned = False

print(f'\n{"="*70}')
print(f'OVERALL: {"✓ ALL 18 MODELS ALIGNED ON CANONICAL INDICES" if all_aligned else "❌ MISALIGNMENT DETECTED"}')
print(f'{"="*70}')

## 7. Commit

In [ ]:
if all_aligned:
    os.chdir(REPO)
    !git add notebooks/04c_shap_canonical.ipynb
    !git add shap_values/*/*_shap_shared_meta.json
    !git add shap_values/*/canonical_*_idx.npy
    !git status --short
    !git commit -m 'Notebook 04c: SHAP recompute on canonical shared samples (18/18 aligned for Krishna agreement analysis)'
    !git push origin main
else:
    print('⚠ Alignment failed — not committing')